In [ ]:
import json
import re
from typing import List, Dict, Optional, Tuple

# ── Schema: defines what the output MUST look like ──────────────────

VALID_INTENTS = ["billing", "cancellation", "technical", "account_change", "general_inquiry"]
VALID_SENTIMENTS = ["positive", "neutral", "frustrated", "angry"]

SCHEMA = {
    "primary_intent": VALID_INTENTS,
    "sub_intent": str,           # free text, e.g. "dispute charge"
    "sentiment": VALID_SENTIMENTS,
    "key_entities": {
        "account_id": "string or null",
        "dollar_amount": "string or null",
        "product": "string or null",
    }
}


# ── Build prompt ────────────────────────────────────────────────────

def build_prompt(turns: List[Dict]) -> str:
    transcript = "\n".join(f"{t['role'].upper()}: {t['text']}" for t in turns)

    return f"""Extract structured info from this conversation.

Output ONLY valid JSON with these fields:
- "primary_intent": one of {VALID_INTENTS}
- "sub_intent": short phrase describing the specific issue
- "sentiment": one of {VALID_SENTIMENTS}
- "key_entities": {{"account_id": str|null, "dollar_amount": str|null, "product": str|null}}

Rules:
- Only include entities explicitly stated in the transcript.
- If something isn't mentioned, use null.

Conversation:
{transcript}

JSON:"""


# ── Parse + validate (the anti-hallucination layer) ─────────────────

def parse_response(raw: str, turns: List[Dict]) -> Tuple[Optional[Dict], List[str]]:
    errors = []

    # Strip markdown fences
    text = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    text = re.sub(r'\s*```$', '', text).strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None, ["Failed to parse JSON"]

    # 1. Validate enums
    if parsed.get("primary_intent") not in VALID_INTENTS:
        errors.append(f"Invalid intent: {parsed.get('primary_intent')}")
        parsed["primary_intent"] = "general_inquiry"  # safe fallback

    if parsed.get("sentiment") not in VALID_SENTIMENTS:
        errors.append(f"Invalid sentiment: {parsed.get('sentiment')}")
        parsed["sentiment"] = "neutral"

    # 2. Ground-check entities against transcript (anti-hallucination)
    transcript_text = " ".join(t["text"] for t in turns)
    entities = parsed.get("key_entities", {})

    for field in ["account_id", "dollar_amount"]:
        val = entities.get(field)
        if val and val not in transcript_text:
            errors.append(f"Hallucinated {field}: '{val}' not in transcript")
            entities[field] = None  # remove the hallucination

    return parsed, errors


# ── End-to-end extraction ───────────────────────────────────────────

def extract(turns: List[Dict]) -> Dict:
    prompt = build_prompt(turns)
    raw = mock_llm_call(prompt)  # swap for real API call
    result, errors = parse_response(raw, turns)

    if result is None:
        return {"error": "Failed to parse", "details": errors}

    result["_validation_errors"] = errors
    return result


# ── Mock LLM for testing ───────────────────────────────────────────

def mock_llm_call(prompt: str) -> str:
    if "$47.99" in prompt:
        return json.dumps({
            "primary_intent": "billing",
            "sub_intent": "unauthorized charge dispute",
            "sentiment": "frustrated",
            "key_entities": {"account_id": "AC-7842961", "dollar_amount": "$47.99", "product": "premium data add-on"}
        })
    return json.dumps({
        "primary_intent": "general_inquiry",
        "sub_intent": "unknown",
        "sentiment": "neutral",
        "key_entities": {"account_id": None, "dollar_amount": None, "product": None}
    })


# ── Test it ─────────────────────────────────────────────────────────

if __name__ == "__main__":
    turns = [
        {"role": "customer", "text": "I was charged $47.99 for a premium data add-on I never ordered."},
        {"role": "agent", "text": "I see that on account AC-7842961. I'll remove it and refund $47.99."},
    ]

    result = extract(turns)
    print(json.dumps(result, indent=2))